In [10]:
import sklearn.decomposition as skd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, HDBSCAN, KMeans
from sklearn.neighbors import KNeighborsClassifier
import glob, os

In [11]:
def train_clustering_for_position_space(base_data):
    clustering = KMeans(n_clusters=25, random_state=42)
    clustering.fit(base_data)
    return clustering

def train_clustering_for_per_position_space(data):
    clustering = HDBSCAN(min_cluster_size=3)
    clustering.fit(data)
    return clustering

def train_kN_neighbors_classifier(base_data, clustering):
    # filter out noise points
    mask = clustering.labels_ != -1
    clean_data = base_data[mask]
    clean_labels = clustering.labels_[mask]
    
    #train kNN classifier
    knn = KNeighborsClassifier(n_neighbors=1) 
    knn.fit(clean_data, clean_labels)
    return knn
    
def predict_cluster_membership(knn, to_predict):
    return knn.predict(to_predict)

def get_cluster_id_for_position(data_for_position, knn):
    # used to test if clusters represent a single position
    matches = np.zeros(25, dtype=int)
    for point in data_for_position:
        predicted_cluster = predict_cluster_membership(knn, [point])[0] # 0 needed here because return is an arrray
        if predicted_cluster == -1: # only count if not noise
            print("Warning: point classified as noise, skipping")
        else:
            matches[predicted_cluster] += 1
    same_cluster_ratio = np.max(matches)/len(data_for_position)
    return np.argmax(matches), same_cluster_ratio

def get_position_map(knn):
    position_map = {}
    for file in glob.glob(os.path.join("data/positionFiles", "*.csv")):
        data = pd.read_csv(file)
        position_name = os.path.basename(file).split(".")[0]
        cluster_id, ratio = get_cluster_id_for_position(data.values, knn)
        position_map[position_name] = (cluster_id, ratio)
        
    print(position_map)
    return position_map

In [ ]:
base_data = pd.read_csv("data/pca_transformed.csv").values
clustering = train_clustering_for_position_space(base_data)
knn = train_kN_neighbors_classifier(base_data, clustering)
position_map = get_position_map(knn)
